In [ ]:
import torch
import plotly.express as px
import plotly.graph_objects as go
from pprint import pprint
import pandas
from copy import copy
from plotly.subplots import make_subplots
from matplotlib.gridspec import GridSpec

outputs = torch.load("../../databases/results/posebusters_all_df.pt", weights_only=False)
result_dfs = outputs["bust_df"]
tools_ordered = ["Vina"]
result_dfs = {tool: result_dfs[tool] for tool in tools_ordered}
cols = [
    "ligand_detail",
    "blind_docking",
    "bond_lengths",
    "bond_angles",
    "aromatic_ring_maximum_distance_from_plane",
    "double_bond_maximum_distance_from_plane",
    "internal_steric_clash",
    "minimum_distance_to_protein",
    "rmsd",
    "centroid_distance",
    "volume_overlap_protein"
]

targets = [
    "rmsd",
    "centroid_distance",
    "bond_lengths",
    "bond_angles",
    "aromatic_ring_maximum_distance_from_plane",
    "double_bond_maximum_distance_from_plane",
    "internal_steric_clash",
    "minimum_distance_to_protein",
]
final_df_data = {}
for c in targets:
    final_df_data[c] = []
    final_df_data[f"{c}_std"] = []

tool_names = []

# cols for final df
tools = []
counts = []
failed_names = []
merged_dfs = []
for i, (tool, full_results) in enumerate(result_dfs.items()):
    print(full_results.tool)
    assert all(full_results.tool == tool), f"not all results come from {tool}"
    tool_names.append(tool)
    results = full_results.loc[
        (full_results.mol_pred_loaded) &
        (full_results.mol_true_loaded) &
        (full_results.mol_cond_loaded) &  # generally, all types of molecules are loaded
        (~full_results["rmsd_≤_2å"].isna()) &
        (~full_results.rmsd.isna()) &
        (~full_results.centroid_distance.isna())&
        (full_results.sanitization), 
        cols
    ]

file                                                 molecule                                             position
../../databases/Vina/outputs/7M41_YQG/7M41_YQG1.sdf  ../../databases/Vina/outputs/7M41_YQG/7M41_YQG1.pdb  0           Vina
../../databases/Vina/outputs/8BPL_CP/8BPL_CP1.sdf    ../../databases/Vina/outputs/8BPL_CP/8BPL_CP1.pdb    0           Vina
../../databases/Vina/outputs/7A9E_R4W/7A9E_R4W1.sdf  ../../databases/Vina/outputs/7A9E_R4W/7A9E_R4W1.pdb  0           Vina
../../databases/Vina/outputs/7MGT_ZD4/7MGT_ZD41.sdf  ../../databases/Vina/outputs/7MGT_ZD4/7MGT_ZD41.pdb  0           Vina
../../databases/Vina/outputs/7BCP_GCO/7BCP_GCO1.sdf  ../../databases/Vina/outputs/7BCP_GCO/7BCP_GCO1.pdb  0           Vina
                                                                                                                      ... 
../../databases/Vina/outputs/7UF2_5SP/7UF2_5SP1.sdf  ../../databases/Vina/outputs/7UF2_5SP/7UF2_5SP1.pdb  0           Vina
../../databases/Vina/out

In [ ]:
from scipy import stats
def compute_significance(ref, test):
    res = stats.ttest_ind(ref, test, equal_var=False)
    p_value = res.pvalue
    significance = ""
    if p_value < 0.001:
        significance = "***"
    elif p_value < 0.01:
        significance =  "**"
    elif p_value < 0.05:
        significance =  "*"
    return significance

In [ ]:
# Violin plot
tools = list(result_dfs.keys())
is_TBD = True
fig_data = []
result_dfs_orders = {
    "Boltz-2": result_dfs["Boltz-2"],
    "AlphaFold3": result_dfs["AlphaFold3"],
    "DynamicBind": result_dfs["DynamicBind"],
    "DiffDock": result_dfs["DiffDock"],
    "TankBind": result_dfs["TankBind"],
    "NeuralPLexer": result_dfs["NeuralPLexer"],
    "EquiBind": result_dfs["EquiBind"],
    "DPL": result_dfs["DPL"],
    "Vina": result_dfs["Vina"],
}
vina_df = result_dfs["Vina"]
vina_pbd = vina_df.loc[
    (vina_df.blind_docking == is_TBD) &
    (vina_df.rmsd < 20) &
    (vina_df.centroid_distance < 20)
]
rmsd_significances = []
centroid_significances = []
for i,(tool, df_full) in enumerate(result_dfs_orders.items()):
    add_TBD = " TBD" if tool == "Vina" else ""
    df = copy(df_full.loc[
        (df_full.blind_docking == is_TBD) &
        (df_full.rmsd < 20) &
        (df_full.centroid_distance < 20)
    ])
    fig_data.extend([
        go.Violin(
            y=df.rmsd.values,
            x = [tool + add_TBD]*(len(df)),
            side="negative",
            legendgroup="RMSD",
            name="RMSD",
            showlegend=i==0,
            points="all",
            jitter=0.1,
            line_color="blue",
        ),
        go.Violin(
            y=df.centroid_distance.values,
            x = [tool + add_TBD]*len(df),
            side= "positive",
            name="Centroid distance",
            legendgroup="Centroid distance",
            line_color="red",
            showlegend=i==0,
            points=False,
        )
    ])
    if tool == "Vina":
        rmsd_significances.append("")
        centroid_significances.append("")
        tool = "Vina PBD"
        df = copy(df_full.loc[
            (df_full.blind_docking == False) &
            (df_full.rmsd < 20) &
            (df_full.centroid_distance < 20)
        ])
        fig_data.extend([
            go.Violin(
                y=df.rmsd.values,
                x = [tool]*(len(df)),
                side="negative",
                legendgroup="RMSD",
                name="RMSD",
                showlegend=False,
                points="all",
                jitter=0.1,
                line_color="blue",
            ),
            go.Violin(
                y=df.centroid_distance.values,
                x = [tool]*len(df),
                side= "positive",
                name="Centroid distance",
                legendgroup="Centroid distance",
                line_color="red",
                showlegend=False,
                points=False,
            )
        ])
    else:
        rmsd_significances.append(
            compute_significance(vina_pbd.rmsd.values, df.rmsd.values)
        )
        centroid_significances.append(
            compute_significance(vina_pbd.centroid_distance.values, df.centroid_distance.values)
        )
print("Rmsd significances:", rmsd_significances)
print("Centroid significances:", centroid_significances)
annotations = []
annotation_y = 20
for i, sig in enumerate(rmsd_significances):
    annotations.append(dict(
        x=i-0.2,  # Adjust based on your x mapping
        y=annotation_y,
        text=sig,
        showarrow=False,
        font=dict(size=20, color="blue"),
        xanchor="center",
        yanchor="bottom"
    ))

# Add Centroid distance significance annotations (top-right side of each violin)
for i, sig in enumerate(centroid_significances):
    annotations.append(dict(
        x=i+0.2,  # Adjust based on your x mapping
        y=annotation_y,  # Slightly lower for centroid to avoid overlap
        text=sig,
        showarrow=False,
        font=dict(size=20, color="red"),
        xanchor="center",
        yanchor="bottom"
    ))
fig = go.Figure(fig_data)
fig.add_vline(x=7.3, line_width=5, line_color="green")
fig.update_yaxes(range=[-5,25], title={"text": "Ångström"})
# fig.update_xaxes(showticklabels=False, row=1, col=1)
fig.update_layout(
    font_size=23,
    violingap=0, violinmode="overlay",
    margin=dict(t=0,b=0,l=50,r=0),
    width=1300,
    height=450,
    legend = {
        "orientation": "h",
        "yanchor": "bottom",
        "xanchor": "right",
        "y":1.02,
        "x":1
    },
    annotations=annotations,
)
fig.show()
fig.write_image("./figures/violin.jpg", width=1300, height=450, scale=2)
fig_data = []
for i,(tool, df_full) in enumerate(result_dfs.items()):
    df = df_full.loc[df_full.blind_docking == True]
    fig_data.extend([
        go.Violin(
            y=df.rmsd.values,
            x = [tool]*len(df),
            side="negative",
            legendgroup="RMSD",
            name="RMSD",
            showlegend=i==0,
            points="all",
            jitter=0.1,
            line_color="blue",
            quartilemethod="inclusive"
        ),
        go.Violin(
            y=df.centroid_distance.values,
            x = [tool]*len(df),
            side= "positive",
            name="Centroid distance",
            legendgroup="Centroid distance",
            line_color="red",
            showlegend=i==0,
            quartilemethod="inclusive"
        )
    ])
fig = go.Figure(fig_data)
fig.update_yaxes(title={"text": "Ångström"}, range=[-10,150])
fig.update_traces(quartilemethod="exclusive") # or "inclusive", or "linear" by default
fig.update_layout(
    showlegend=False,
    margin=dict(b=0,t=0,l=0,r=0),
    font={"size": 23},
)
fig.show()
fig.write_image("./figures/supplementary_violin_TBD.jpg", width=1300, height=500, scale=2)
fig_data = []
for i,(tool, df_full) in enumerate(result_dfs.items()):
    df = df_full.loc[df_full.blind_docking == False]
    fig_data.extend([
        go.Violin(
            y=df.rmsd.values,
            x = [tool]*len(df),
            side="negative",
            legendgroup="RMSD",
            name="RMSD",
            showlegend=i==0,
            points="all",
            jitter=0.1,
            line_color="blue",
            quartilemethod="inclusive"
        ),
        go.Violin(
            y=df.centroid_distance.values,
            x = [tool]*len(df),
            side= "positive",
            name="Centroid distance",
            legendgroup="Centroid distance",
            line_color="red",
            showlegend=i==0,
            quartilemethod="inclusive"
        )
    ])
fig = go.Figure(fig_data)
fig.update_yaxes(title={"text": "Ångström"}, range=[-10,150])
fig.update_traces(quartilemethod="exclusive") # or "inclusive", or "linear" by default
fig.update_layout(
    showlegend=False,
    margin=dict(b=0,t=0,l=0,r=0),
    font={"size": 23},
)
fig.show()
fig.write_image("./figures/supplementary_violin_PBD.jpg", width=1300, height=500, scale=2)

In [ ]:
# Violin plot
tools = list(result_dfs.keys())
is_TBD = False
fig_data = []
result_dfs_orders = {
    "Boltz-2": result_dfs["Boltz-2"],
    "AlphaFold3": result_dfs["AlphaFold3"],
    "DynamicBind": result_dfs["DynamicBind"],
    "DiffDock": result_dfs["DiffDock"],
    "TankBind": result_dfs["TankBind"],
    "NeuralPLexer": result_dfs["NeuralPLexer"],
    "EquiBind": result_dfs["EquiBind"],
    "DPL": result_dfs["DPL"],
    "Vina": result_dfs["Vina"],
}
vina_df = result_dfs["Vina"]
vina_pbd = vina_df.loc[
    (vina_df.blind_docking == is_TBD) &
    (vina_df.rmsd < 20) &
    (vina_df.centroid_distance < 20)
]
rmsd_significances = ["", ""]
centroid_significances = ["", ""]
for i,(tool, df_full) in enumerate(result_dfs_orders.items()):
    add_TBD = " TBD" if tool == "Vina" else ""
    df = copy(df_full.loc[
        (df_full.blind_docking == True) &
        (df_full.rmsd < 20) &
        (df_full.centroid_distance < 20)
    ])
    fig_data.extend([
        go.Violin(
            y=df.rmsd.values,
            x = [tool + add_TBD]*(len(df)),
            side="negative",
            legendgroup="RMSD",
            name="RMSD",
            showlegend=i==0,
            points="all",
            jitter=0.1,
            line_color="blue",
        ),
        go.Violin(
            y=df.centroid_distance.values,
            x = [tool + add_TBD]*len(df),
            side= "positive",
            name="Centroid distance",
            legendgroup="Centroid distance",
            line_color="red",
            showlegend=i==0,
            points=False,
        )
    ])
    if tool == "Vina":
        tool = "Vina PBD"
        df = copy(df_full.loc[
            (df_full.blind_docking == False) &
            (df_full.rmsd < 20) &
            (df_full.centroid_distance < 20)
        ])
        fig_data.extend([
            go.Violin(
                y=df.rmsd.values,
                x = [tool]*(len(df)),
                side="negative",
                legendgroup="RMSD",
                name="RMSD",
                showlegend=False,
                points="all",
                jitter=0.1,
                line_color="blue",
            ),
            go.Violin(
                y=df.centroid_distance.values,
                x = [tool]*len(df),
                side= "positive",
                name="Centroid distance",
                legendgroup="Centroid distance",
                line_color="red",
                showlegend=False,
                points=False,
            )
        ])
    else:
        rmsd_significances.append(
            compute_significance(vina_pbd.rmsd.values, df.rmsd.values)
        )
        centroid_significances.append(
            compute_significance(vina_pbd.centroid_distance.values, df.centroid_distance.values)
        )
print("Rmsd significances:", rmsd_significances)
print("Centroid significances:", centroid_significances)
annotations = []
annotation_y = 20
for i, sig in enumerate(rmsd_significances):
    annotations.append(dict(
        x=i-0.2,  # Adjust based on your x mapping
        y=annotation_y,
        text=sig,
        showarrow=False,
        font=dict(size=20, color="blue"),
        xanchor="center",
        yanchor="bottom"
    ))

# Add Centroid distance significance annotations (top-right side of each violin)
for i, sig in enumerate(centroid_significances):
    annotations.append(dict(
        x=i+0.2,  # Adjust based on your x mapping
        y=annotation_y,  # Slightly lower for centroid to avoid overlap
        text=sig,
        showarrow=False,
        font=dict(size=20, color="red"),
        xanchor="center",
        yanchor="bottom"
    ))
fig = go.Figure(fig_data)
fig.add_vline(x=7.3, line_width=5, line_color="green")
fig.update_yaxes(range=[-5,25], title={"text": "Ångström"})
# fig.update_xaxes(showticklabels=False, row=1, col=1)
fig.update_layout(
    font_size=23,
    violingap=0, violinmode="overlay",
    margin=dict(t=0,b=0,l=50,r=0),
    width=1300,
    height=450,
    legend = {
        "orientation": "h",
        "yanchor": "bottom",
        "xanchor": "right",
        "y":1.02,
        "x":1
    },
    annotations=annotations,
)
fig.show()
fig.write_image("./figures/violin_pbd.jpg", width=1300, height=450, scale=2)

In [ ]:
from copy import copy
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create subplots: one row per tool
result_dfs_orders = {t['tool']:t['df'] for t in merged_dfs}
n_tools = len(result_dfs_orders)
is_TBD = True

# Create subplot figure with n_tools rows and 1 column
fig = make_subplots(
    rows=5, cols=2,
    # subplot_titles=list(result_dfs_orders.keys()),
    shared_xaxes=False,
    vertical_spacing=0.1,
    horizontal_spacing=0.05,
)

# Track which legends have been added
rmsd_legend_added = False
centroid_legend_added = False
annotations = []

# Loop through each tool (row in subplot)
for row_idx, (tool, df_full) in enumerate(result_dfs_orders.items()):
    row = (row_idx // 2) + 1
    col = (row_idx % 2) + 1
    # Process each enzyme class
    tool_rmsd_significances = {}
    tool_centroid_significances = {}
    class_A = copy(df_full.loc[
        (df_full.blind_docking == is_TBD) &
        (df_full.rmsd < 20) &
        (df_full.centroid_distance < 20) &
        (df_full.group == "A")
    ])
    for enz_class in ["A", "B", "C"]:
        df = copy(df_full.loc[
            (df_full.blind_docking == is_TBD) &
            (df_full.rmsd < 20) &
            (df_full.centroid_distance < 20) &
            (df_full.group == enz_class)
        ])
        # compute significance per class, per tool
        tool_rmsd_significances[enz_class] = compute_significance(
            class_A.rmsd.values, df.rmsd.values
        )
        tool_centroid_significances[enz_class] = compute_significance(
            class_A.centroid_distance.values, df.centroid_distance.values
        )
        
        # Create x-values for this class within the tool
        x_values = [f"{enz_class}"] * len(df)
        
        # Add RMSD violin (negative side)
        fig.add_trace(
            go.Violin(
                y=df.rmsd.values,
                x=x_values,
                side="negative",
                legendgroup="RMSD",
                name="RMSD",
                showlegend=not rmsd_legend_added,
                # points="all",
                # jitter=0.1,
                line_color="blue",
                scalegroup=tool,  # Group violins within each tool
            ),
            row=row, col=col
        )
        
        # Add Centroid distance violin (positive side)
        fig.add_trace(
            go.Violin(
                y=df.centroid_distance.values,
                x=x_values,
                side="positive",
                legendgroup="Centroid distance",
                name="Centroid distance",
                showlegend=not centroid_legend_added,
                points=False,
                line_color="red",
                scalegroup=tool,  # Group violins within each tool
            ),
            row=row, col=col
        )
        
        # Update legend flags
        if not rmsd_legend_added:
            rmsd_legend_added = True
        if not centroid_legend_added:
            centroid_legend_added = True

    # Add significance annotations to each subplot
    y_position = 23
    fig.add_annotation(
        x=.5,
        y=1.1,  # Position above the subplot
        xref=f"x domain",
        yref=f"y domain",
        row=row,
        col=col,
        text=tool,
        showarrow=False,
        font=dict(size=14, color="black", weight="bold"),
        xanchor="center",
        yanchor="bottom"
    )
    for i,enz_class in enumerate(["A", "B", "C"]):
        rmsd_sig = tool_rmsd_significances[enz_class]
        if rmsd_sig:  # Only add if there is significance
            fig.add_annotation(
                x=i-0.15,  # Position on left side of subplot
                y=y_position,
                xref=f"x{row * 2 - 1 if col == 1 else row * 2}",
                yref=f"y{row * 2 - 1 if col == 1 else row * 2}",
                text=rmsd_sig,
                showarrow=False,
                font=dict(size=10, color="blue"),
                xanchor="left",
                yanchor="top"
            )

    # Add Centroid significances for each enzyme class
    for i,enz_class in enumerate(["A", "B", "C"]):
        centroid_sig = tool_centroid_significances[enz_class]
        if centroid_sig:  # Only add if there is significance
            fig.add_annotation(
                x=i+0.15,  # Position on right side of subplot
                y=y_position,
                xref=f"x{row * 2 - 1 if col == 1 else row * 2}",
                yref=f"y{row * 2 - 1 if col == 1 else row * 2}",
                text=centroid_sig,
                showarrow=False,
                font=dict(size=16, color="red"),
                xanchor="right",
                yanchor="top"
            )

# Update layout for all subplots
fig.update_yaxes(range=[-5, 25], title={"text": "Ångström"}, col=1)
fig.update_xaxes(range=[-0.31, 2.4])

# Update overall figure layout
fig.update_layout(
    font_size=14,  # Reduced slightly to fit better in subplots
    violingap=0, 
    violinmode="overlay",
    margin=dict(t=0, b=0, l=0, r=0),  # Adjusted margins
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "xanchor": "right",
        "y": 1.05,
        "x": 1,
        "font": {"size": 14}
    },
    showlegend=True,
    # annotations = annotations
)
fig.write_image("./figures/violin_per_class.jpg", height=600, width=500, scale=2)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Your existing data gathering code remains the same
tools = list(result_dfs_orders.keys())
hitrates = {
    "TBD": {
        "2": {
            "full": [],
            "A": [],
            "B": [],
            "C": []
        },
        "5": {
            "full": [],
            "A": [],
            "B": [],
            "C": []
        },
    },
    "PBD": {
        "2": {
            "full": [],
            "A": [],
            "B": [],
            "C": []
        },
        "5": {
            "full": [],
            "A": [],
            "B": [],
            "C": []
        },
    },
}
enz_class_proportions = {
    "TBD": {
        "A": [],
        "B": [],
        "C": []
    },
    "PBD": {
        "A": [],
        "B": [],
        "C": []
    }
}
enz_class_counts = {
    "TBD": {
        "A": [],
        "B": [],
        "C": []
    },
    "PBD": {
        "A": [],
        "B": [],
        "C": []
    }
}


merged_dfs = [
    [merged_df for merged_df in merged_dfs if merged_df['tool']==tool][0]
for tool in tools]
for i, (df_data) in enumerate(merged_dfs):
    all_df = df_data["df"]
    tool = df_data["tool"]
    for docking_mode in ["TBD", "PBD"]:
        is_TBD = docking_mode == "TBD"
        docking_regime_df = all_df.loc[all_df.blind_docking == is_TBD]
        df = all_df.loc[
            (all_df.blind_docking == is_TBD)
        ]
        for threshold in ["2", "5"]:
            hitcount = torch.tensor(df.rmsd.values < int(threshold), dtype=torch.bool).float().sum()
            hitrate = (hitcount/df.shape[0])*100
            hitrates[docking_mode][threshold]["full"].append(hitrate.item())
        for enz_class in ["A", "B", "C"]:
            df = all_df.loc[
                (all_df.blind_docking == is_TBD) &
                (all_df.group == enz_class)
            ]
            enz_count = df.shape[0]
            enz_proportion = enz_count/docking_regime_df.shape[0]*100
            enz_class_counts[docking_mode][enz_class].append(enz_count)
            enz_class_proportions[docking_mode][enz_class].append(enz_proportion)
            for threshold in ["2","5"]:
                hitcount = torch.tensor(df.rmsd.values < int(threshold), dtype=torch.bool).float().sum()
                hitrate = (hitcount/df.shape[0])*100
                hitrates[docking_mode][threshold][enz_class].append(hitrate.item())


# Colors for enzyme classes
class_colors = {
    'A': "#4EC218",  # Coral red
    'B': "#8AA6FF",  # Turquoise
    'C': "#0026FF"   # Sky blue
}
def plot_docking_analysis(hitrates, docking_regime):
    """
    Plot docking analysis results in 4 subplots.
    
    Parameters:
    -----------
    hitrates : dict
        Dictionary with structure {'TBD':{'A':{'2':[], '5':[]}, ...}, 'PBD':{...}}
    enz_class_proportions : dict
        Dictionary with structure {'TBD':{'A':[], 'B':[], 'C':[]}, 'PBD':{...}}
    tools : list, optional
        List of tool names. If None, inferred from data length.
    """
    

    # ============================================================
    # Subplot 3: Hit Rates at 2Å Threshold
    # ============================================================
    fig, (ax3,ax4) = plt.subplots(2,1,figsize=(9, 6))
    
    # Grouped bars for TBD vs PBD for each tool
    grouped_hitrates = {enz_class: hitrates[docking_regime]["2"][enz_class] for enz_class in ["A", "B", "C","full"]}
    if docking_regime == "TBD":
        for enz_class in ["A","B","C","full"]:
            vina_PBD_enz = hitrates["PBD"]["2"][enz_class][-1]
            grouped_hitrates[enz_class].append(vina_PBD_enz)
        tools[-1] = "Vina TBD"
        tools.append("Vina PBD")
    ax3.grouped_bar(
        grouped_hitrates, 
        tick_labels=tools, 
        group_spacing=1,
        colors=[class_colors["A"],class_colors["B"],class_colors["C"], "black"])
    
    # Add dummy bars for legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=class_colors['A'], edgecolor='white', label='Class A'),
        Patch(facecolor=class_colors['B'], edgecolor='white', label='Class B'),
        Patch(facecolor=class_colors['C'], edgecolor='white', label='Class C'),
        Patch(facecolor="black", edgecolor='white', label='All classes'),
    ]
    
    ax3.set_ylabel('Hit Rate (%)', fontsize=14)
    ax3.set_xticklabels(tools, rotation=45, ha='right')
    ax3.legend(
        handles=legend_elements, 
        fontsize=12, 
        loc='upper right', 
        ncol=4,
        handletextpad=0.3
    )
    ax3.tick_params(axis="both", labelsize=14)
    ax3.grid(True, alpha=0.3, axis='y')
    ax3.axvline(x=7.4, color="green",linewidth=3)
    ax3.set_ylim([0,60])

    # ============================================================
    # Subplot 4: Hit Rates at 5Å Threshold
    # ============================================================
    
    # Grouped bars for TBD vs PBD for each tool
    grouped_hitrates = {enz_class: hitrates[docking_regime]["5"][enz_class] for enz_class in ["A", "B", "C", "full"]}
    if docking_regime == "TBD":
        for enz_class in ["A","B","C","full"]:
            vina_PBD_enz = hitrates["PBD"]["5"][enz_class][-1]
            grouped_hitrates[enz_class].append(vina_PBD_enz)
    ax4.grouped_bar(
        grouped_hitrates, 
        tick_labels=tools, 
        group_spacing=1,
        colors=[class_colors["A"],class_colors["B"],class_colors["C"], "black"])
    # ax4.set_xticks(x)
    ax4.axvline(x=7.4, color="green",linewidth=3)
    ax4.set_ylabel('Hit Rate (%)', fontsize=14)
    ax4.set_xticklabels(tools, rotation=45, ha='right')
    ax4.tick_params(axis="both", labelsize=14)
    ax4.grid(True, alpha=0.3, axis='y')
    ax4.set_ylim([0,60])

    annotation_labels = ['A', 'B']
    annotation_positions = [
        (-0.09, 0.95),  # A - top-left
        (-0.09, 0.95),  # B - top-right
    ]
    axes = [ax3, ax4]
    for i, (ax, (x_pos, y_pos), label) in enumerate(zip(axes, annotation_positions, annotation_labels)):
        ax.text(x_pos, y_pos, label, transform=ax.transAxes, 
                fontsize=16, fontweight='bold', va='bottom', ha='right')
    
    # Adjust layout
    fig.subplots_adjust(
        wspace=0.2,
        hspace=0.25,
        bottom=0.35,
        top=0.95,
        left=0.09,
        right=0.99
    )
    
    # Save figure
    fig.tight_layout()
    fig.savefig(f"figures/hitrates_{docking_regime}.jpg", dpi=600)
    fig.show()
    
    return fig

fig = plot_docking_analysis(hitrates, "TBD")
tools = list(result_dfs_orders.keys())
fig = plot_docking_analysis(hitrates, "PBD")

In [ ]:
# Draw the data distribution graph

# prepare data:
df = load_db("../../databases/QBioLiP/qbiolip.csv", "../../databases/ec_uniprot_ids.txt")
df = df.loc[df.target]
df["num_atoms"] = df.lig_atoms.str.len()
classes = ["A", "B", "C"]
alphas = [1,0.5,0.5]
colors = []

# get residue counts:
def get_res_count(rec_file_id, tbd):
    if tbd:
        rec_path = f"../../databases/QBioLiP/nr_rec/{rec_file_id}.pdb"
    else:
        rec_path = f"../../databases/QBioLiP/nr_rec_trimed/{rec_file_id}.pdb"
    with open(rec_path, "r") as rec_f:
        res_indices = set([l[22:26].strip() for l in rec_f if l[0:5].strip() == "ATOM"])
    return len(res_indices)

df["tbd_res_count"] = [get_res_count(assembly_id, True) for assembly_id in df.assembly_id.tolist()]
df["pbd_res_count"] = [get_res_count(ligand_detail, False) for ligand_detail in df.ligand_detail.tolist()]

# Build subplots:
fig, ((ax0, ax1), (ax2, ax3), (ax4, ax5)) = plt.subplots(3,2, figsize=(10,8))

ax0.hist(
    [df.loc[(df.group == enz_class)].tbd_res_count.values for enz_class in classes],
    label=classes, 
    bins=20,
    color = [class_colors[enz_class] for enz_class in classes]
)
ax0.legend(title="Enzyme class", ncols=3)
ax0.set_ylabel("Count", fontsize=12)
ax0.set_xlabel("n residues", fontsize=12)
ax0.tick_params(axis="both", labelsize=12)
ax1.hist(
    [df.loc[df.group == enz_class].pbd_res_count.values for enz_class in classes],
    label=classes, 
    bins=20,
    color = [class_colors[enz_class] for enz_class in classes]
)
ax1.set_xlabel("n residues", fontsize=12)
ax1.tick_params(axis="both", labelsize=12)

ax2.hist(
    [df.loc[df.group == enz_class].num_atoms.values for enz_class in classes],
    label=classes, 
    bins=20,
    color = [class_colors[enz_class] for enz_class in classes]
)
ax2.set_xlabel("n substrate heavy atoms", fontsize=12)
ax2.set_ylabel("Count", fontsize=12)
ax2.tick_params(axis="both", labelsize=12)

# plot stoichiometry distribution:
df["stoichiometry"] = [s.split(" ")[0] for s in df.stoichiometry.values]
stoich_x = [df.loc[df.group == enz_class].stoichiometry.value_counts().index.tolist() for enz_class in classes]
stoich_x = [label for all_label in stoich_x for label in all_label]
stoich_y = [df.loc[df.group == enz_class].stoichiometry.value_counts().values.tolist() for enz_class in classes]

data = {
    "A": stoich_y[0],
    "B": stoich_y[1],
    "C": stoich_y[2]
}
category_name = {
    "Monomer": "Monomer",
    "Hetero": "Heteromer"
}
groups = list(data.keys())
categories = list(set(stoich_x))

x = np.arange(len(categories))
width = 0.20
spacing = 0.05
for i, (group, values) in enumerate(data.items()):
    offset = (i - (len(groups)-1)/2) * (width + spacing)
    ax3.bar(
        x+offset, values, width, label=group, 
        color=class_colors[group]
    )
ax3.set_xticks([0,1])
ax3.set_xticklabels([category_name[c] for c in categories], fontsize=12)
ax3.set_ylabel("Count (log scale)", fontsize=12)
ax3.set_yscale("log")

# Prepare data for stacked bars
n_tools = len(tools)
x = np.arange(n_tools)
bar_width = 0.4
offset = bar_width / 2

# Get proportions for each enzyme class
bottom = np.zeros(n_tools)
for i, tool in enumerate(tools):
    # TBD bar (stacked by enzyme class)
    bottom = 0
    for enz_class in ['A', 'B', 'C']:
        # Get hitrate for this tool, enzyme class, and threshold
        proportion = enz_class_proportions['TBD'][enz_class][i]
        
        bars = ax4.bar(i - offset, proportion, bar_width,
                        color=class_colors[enz_class],
                        edgecolor='white', linewidth=0.5,
                        alpha=0.8,
                        bottom=bottom)
        bottom += proportion

    bottom = 0
    for enz_class in ['A', 'B', 'C']:
        # Get hitrate for this tool, enzyme class, and threshold
        proportion = enz_class_proportions['PBD'][enz_class][i]
        
        bars = ax4.bar(i + offset, proportion, bar_width,
                        color=class_colors[enz_class],
                        edgecolor='white', linewidth=0.5,
                        alpha=0.8,
                        hatch="//",
                        bottom=bottom)
        bottom += proportion
    

ax4.set_ylabel('Enzyme class (%)', fontsize=12)
# ax4.set_xticks(x)
# ax4.set_xticklabels(tools, rotation=45, ha='right')
ax4.tick_params(axis='both', labelsize=12)
ax4.grid(True, alpha=0.3, axis='y')
ax4.set_ylim([0, 100])
ax4.set_xticks(list(range(n_tools)))
ax4.set_xticklabels(tools, rotation=35, ha="right")

# Get counts for each enzyme class
bottom = np.zeros(n_tools)
for i, tool in enumerate(tools):
    # TBD bar (stacked by enzyme class)
    bottom = 0
    for enz_class in ['A', 'B', 'C']:
        # Get count for this tool, enzyme class
        count = enz_class_counts['TBD'][enz_class][i]
        
        bars = ax5.bar(i - offset, count, bar_width,
                        color=class_colors[enz_class],
                        edgecolor='white', linewidth=0.5,
                        alpha=0.8,
                        bottom=bottom)
        bottom += count

    bottom = 0
    for enz_class in ['A', 'B', 'C']:
        # Get count for this tool, enzyme class
        count = enz_class_counts['PBD'][enz_class][i]
        
        bars = ax5.bar(i + offset, count, bar_width,
                        color=class_colors[enz_class],
                        edgecolor='white', linewidth=0.5,
                        alpha=0.8,
                        hatch="//",
                        bottom=bottom)
        bottom += count
ax5.axhline(y=210, color="r", linestyle="--", linewidth=2, label="All modeled")
ax5.set_ylim([0,270])
ax5.set_ylabel("Complex count", fontsize=12)
ax5.tick_params(axis='both', labelsize=12)
ax5.set_xticks(list(range(n_tools)))
ax5.set_xticklabels(tools, rotation=35, ha="right")
ax5.legend(loc="upper right", fontsize=12)
# Subplot annotations A,B
annotation_positions = [
    (-0.09, 0.95),  # A - top-left
    (-0.09, 0.95),  # B - top-right
    (-0.09, 0.95),  # C - bottom-left
    (-0.09, 0.95),  # D - bottom-right
    (-0.09, 1.02),  # E - bottom-right
    (-0.09, 1.02),  # F - bottom-right
]

annotation_labels = ['A', 'B', "C", "D", "E", "F"]

for i, (ax, (x_pos, y_pos), label) in enumerate(zip((ax0,ax1,ax2,ax3,ax4,ax5), annotation_positions, annotation_labels)):
    ax.text(x_pos, y_pos, label, transform=ax.transAxes, 
            fontsize=16, fontweight='bold', va='bottom', ha='right')

plt.tight_layout()
plt.subplots_adjust(hspace=0.3)
plt.savefig("./figures/data_presentation.jpg", dpi=600)
plt.show()
print("Average tbd res count:", df.tbd_res_count.mean())
print("Average pbd res count:", df.pbd_res_count.mean())

In [ ]:
# info on hq and lq cases:
vina_df = result_dfs["Vina"]
af3_df = result_dfs["Boltz-2"]
print(af3_df.loc[af3_df.rmsd < 2, ["ligand_detail", "rmsd"]].values)
print(vina_df.loc[vina_df.rmsd < 2, ["ligand_detail", "rmsd"]].values)
# af3_hq = af3_df.loc[af3_df.ligand_detail == "8aau_1_LH0_C", ["ligand_detail", "blind_docking", "rmsd"]].values
# print(af3_hq)
# vina_hq = vina_df.loc[vina_df.ligand_detail == "6hyj_2_SEP_E", ["ligand_detail", "blind_docking", "rmsd"]].values
# print(vina_hq)
# vina_lq = vina_df.loc[vina_df.ligand_detail == "1ffy_1_MRC_P", ["ligand_detail", "blind_docking", "rmsd"]].values
# print(vina_lq)
# af3_lq = af3_df.loc[af3_df.ligand_detail == "8jb3_1_NOS_B", ["ligand_detail", "blind_docking", "rmsd"]].values
# print(af3_lq)

In [ ]:
criteria = "volume_overlap_protein"

# retrieve TBD data:
tools_ordered = [
    "Boltz-2", "AlphaFold3", "DynamicBind", 
    "DiffDock", "TankBind", "NeuralPLexer", "EquiBind", "DPL", "Vina"]
results_ordered = {t:[df["df"] for df in merged_dfs if df["tool"]==t][0] for t in tools_ordered}
data_tbd = [df.loc[df.blind_docking, [criteria]].values*100 for df in results_ordered.values()]
vina_pbd_df = results_ordered["Vina"].loc[results_ordered["Vina"].blind_docking == False]
data_tbd.append(vina_pbd_df[criteria].values*100)
data_tbd = [a[~np.isnan(a)] for a in data_tbd]
significances = [compute_significance(d, data_tbd[-1]) for d in data_tbd]

# make the figure:
fig, ax = plt.subplots(figsize=(10,3))

# TBD graph:
bars = ax.boxplot(
    data_tbd,
    patch_artist=True,
    notch=True,
    showfliers=False
)
for i, group_data in enumerate(data_tbd):
    # Add jitter to x-position
    x_jitter = np.random.normal(i+0.6, 0.02, len(group_data))
    ax.scatter(x_jitter, group_data, alpha=0.5, color='blue', s=20, zorder=3)
tools_ordered[-1] = "Vina TBD"
tools_ordered.append("Vina PBD")
ax.set_xticklabels(tools_ordered, rotation=20, fontsize=14)

ax.set_ylabel("Percentage", fontsize=14)
ax.axvline(x=8.4, color="green")
ax.tick_params(axis="y", labelsize=14)
ax.grid(True, axis="y")
for i, significance in enumerate(significances):
    height = 73
    ax.text(
        i+1, 
        height, significance, ha='center', va='bottom', fontsize=13, color="red",
    )

plt.tight_layout()
plt.savefig("./figures/boxplot_volume_overlap.jpg", dpi=600)
plt.show()
tools_ordered = [
    "Boltz-2", "AlphaFold3", "DynamicBind", 
    "DiffDock", "TankBind", "NeuralPLexer", "EquiBind", "DPL", "Vina"]

In [ ]:
fig, axes = plt.subplots(5,2,figsize=(6,7))
axes = axes.flatten()
axes[9].set_visible(False)
enz_classes = ["A", "B", "C"]
for i, (tool, df) in enumerate(results_ordered.items()):
    data = []
    class_A = df.loc[
        (df.group=="A") &
        (df.blind_docking),
    [criteria]].values*100
    class_A = class_A.squeeze(1)
    class_A = class_A[~np.isnan(class_A)]
    for j,enz_class in enumerate(["A", "B", "C"]):
        y = df.loc[
            (df.group==enz_class) &
            (df.blind_docking),
        [criteria]].values*100
        y = y.squeeze(1)
        y = y[~np.isnan(y)]
        data.append(y)
        sign = compute_significance(class_A,y)
        axes[i].text(
            j+1,65, sign, color="red", fontsize=16,
            ha="center",va="bottom"
        )
        x_jitter = np.random.normal(j+0.6, 0.02, len(y))
        axes[i].scatter(x_jitter, y, alpha=0.5, color='blue', s=20, zorder=3)
    axes[i].boxplot(
        data,
        patch_artist=True,
        notch=True,
        showfliers=False
    )
    axes[i].set_ylim([0,100])
    axes[i].set_title(tool, fontweight="bold")
    # axes[i].set_xticks(list(range(len(enz_classes))))
    axes[i].set_xticklabels(enz_classes)
    if i%2==0:
        axes[i].set_ylabel("Percentage", fontsize=12)
    axes[i].tick_params(axis="both", labelsize=12)
    if i == len(results_ordered)-1:
        break

plt.tight_layout()
fig.subplots_adjust(
    hspace=0.5,
    left=0.11,
    right=0.98,
    top=0.97,
    bottom=0.04
)
plt.savefig("./figures/boxplot_volume_overlap_per_class.jpg", dpi=600)